In [1]:
import pandas as pd
import numpy as np
import re
from datetime import datetime


In [2]:
df = pd.read_csv("/kaggle/input/datasets/noorabushawar/browsing-data/raw_browsing_history.csv")

print(f"• Total records: {len(df)}")
print(f"• Columns : {list(df.columns)}")
print(df.head(5).to_string(index=False))


• Total records: 650
• Columns : ['user_name', 'website_url', 'website_name', 'visit_date', 'visit_time']
user_name               website_url website_name visit_date visit_time
     Noor    https://www.kaggle.com       kaggle   19-10-25      20:28
     Noor    https://www.kaggle.com       Kaggle   14-10-25   19:42:21
     Noor     https://www.twitch.tv       Twitch   30-12-25    5:03 AM
    Shahd    https://www.tiktok.com       TikTok   12-11-25   13:22:53
     Sara https://www.wikipedia.org    wikipedia   17-12-25   22:10:26


In [3]:
null_counts  = df.isnull().sum()
print(f"• Null values per column:\n{null_counts[null_counts > 0].to_string() if null_counts.sum() > 0 else '    None found'}")

# Replace empty strings with NaN for uniform handling
df.replace("", np.nan, inplace=True)


• Null values per column:
website_url     12
website_name    52
visit_time      30


In [4]:
before = len(df)
df.drop_duplicates(inplace=True)
df.reset_index(drop=True, inplace=True)
after = len(df)

print(f"• Duplicates removed : {before - after}")
print(f"• Records remaining  : {after}")


• Duplicates removed : 50
• Records remaining  : 600


In [5]:
NAME_MAP = {
    # YouTube
    "youtube": "YouTube", "you tube": "YouTube", "youtube": "YouTube",
    # Google
    "google": "Google", "gooogle": "Google",
    # Instagram
    "instagram": "Instagram", "instragram": "Instagram",
    # Twitter
    "twitter": "Twitter / X", "x": "Twitter / X", "twitter/x": "Twitter / X",
    # Facebook
    "facebook": "Facebook", "facbook": "Facebook",
    # Netflix
    "netflix": "Netflix", "netfix": "Netflix",
    # Reddit
    "reddit": "Reddit", "reddt": "Reddit",
    # Amazon
    "amazon": "Amazon", "amazone": "Amazon",
    # Wikipedia
    "wikipedia": "Wikipedia", "wikepedia": "Wikipedia",
    # GitHub
    "github": "GitHub", "git hub": "GitHub",
    # Stack Overflow
    "stackoverflow": "Stack Overflow", "stack overflow": "Stack Overflow", "stackoverflw": "Stack Overflow",
    # TikTok
    "tiktok": "TikTok", "tik tok": "TikTok",

    # LinkedIn
    "linkedin": "LinkedIn", "linked in": "LinkedIn",
    # Spotify
    "spotify": "Spotify", "spotfy": "Spotify",
    # Canva
    "canva": "Canva", "cnava": "Canva",
    # Coursera
    "coursera": "Coursera", "courcera": "Coursera",
    # Kaggle
    "kaggle": "Kaggle", "kagle": "Kaggle",
    # W3Schools
    "w3schools": "W3Schools", "w3 schools": "W3Schools", "w3school": "W3Schools",
    # ChatGPT
    "chatgpt": "ChatGPT", "chatgpt": "ChatGPT", "chat gpt": "ChatGPT",
    # Twitch
    "twitch": "Twitch", "twtich": "Twitch",
}

def standardize_name(name, url):
    """Standardize website name. If missing, infer from URL."""
    if pd.isna(name):
        # Infer from URL

        if pd.isna(url):
            return np.nan
        domain = re.sub(r"https?://(www\.)?", "", str(url)).split("/")[0].split(".")[0].lower()
        return NAME_MAP.get(domain, domain.capitalize())
    return NAME_MAP.get(str(name).strip().lower(), str(name).strip())

df["website_name"] = df.apply(lambda r: standardize_name(r["website_name"], r["website_url"]), axis=1)

print(f"• Unique site names after cleaning: {df['website_name'].nunique()}")
print(f"• Names: {sorted(df['website_name'].dropna().unique())}")

• Unique site names after cleaning: 20
• Names: ['Amazon', 'Canva', 'ChatGPT', 'Coursera', 'Facebook', 'GitHub', 'Google', 'Instagram', 'Kaggle', 'LinkedIn', 'Netflix', 'Reddit', 'Spotify', 'Stack Overflow', 'TikTok', 'Twitch', 'Twitter / X', 'W3Schools', 'Wikipedia', 'YouTube']


In [6]:
DATE_FMTS = ["%Y-%m-%d", "%d/%m/%Y", "%m-%d-%Y", "%d-%m-%Y", "%m/%d/%Y"]

def parse_date(val):
    if pd.isna(val):
        return np.nan
    for fmt in DATE_FMTS:
        try:
            return datetime.strptime(str(val).strip(), fmt).strftime("%Y-%m-%d")
        except ValueError:
            continue
    return np.nan

df["visit_date"] = df["visit_date"].apply(parse_date)
null_dates = df["visit_date"].isna().sum()
print(f" Date Standardization --> YYYY-MM-DD")
print(f"  • Unparseable dates  : {null_dates}")


 Date Standardization --> YYYY-MM-DD
  • Unparseable dates  : 504


In [7]:
#STANDARDIZE TIMES --> HH:MM:SS (24h)
TIME_FMTS = ["%H:%M:%S", "%H:%M", "%I:%M %p", "%I:%M:%S %p"]

def parse_time(val):
    if pd.isna(val):
        return "00:00:00"   # default to midnight if missing
    for fmt in TIME_FMTS:
        try:
            return datetime.strptime(str(val).strip(), fmt).strftime("%H:%M:%S")
        except ValueError:
            continue
    return "00:00:00"

df["visit_time"] = df["visit_time"].apply(parse_time)

print(f"  • Sample times: {df['visit_time'].head(5).tolist()}")


  • Sample times: ['20:28:00', '19:42:21', '05:03:00', '13:22:53', '22:10:26']


In [8]:
# Drop rows where website_name and URL are both null
before_drop = len(df)
df.dropna(subset=["website_name"], inplace=True)
df.reset_index(drop=True, inplace=True)

print(f"• Rows dropped (no website info) : {before_drop - len(df)}")
print(f"• Remaining records              : {len(df)}")


• Rows dropped (no website info) : 2
• Remaining records              : 598


In [9]:
df["visit_datetime"] = pd.to_datetime(df["visit_date"] + " " + df["visit_time"])
df["day_of_week"]    = df["visit_datetime"].dt.day_name()
df["hour_of_day"]    = df["visit_datetime"].dt.hour
df["month"]          = df["visit_datetime"].dt.strftime("%B")
df["month_num"]      = df["visit_datetime"].dt.month
df["week_of_year"]   = df["visit_datetime"].dt.isocalendar().week.fillna(0).astype(int)

# Time-of-day category
def time_category(hour):
    if 5 <= hour < 12:  return "Morning"
    if 12 <= hour < 17: return "Afternoon"
    if 17 <= hour < 21: return "Evening"
    return "Night"

df["time_of_day"] = df["hour_of_day"].apply(time_category)


print(f"• Columns: day_of_week, hour_of_day, month, week_of_year, time_of_day")


• Columns: day_of_week, hour_of_day, month, week_of_year, time_of_day


In [10]:
# DIM_USER
users = pd.DataFrame({
    "user_id"  : range(1, len(USERS := df["user_name"].unique()) + 1),
    "user_name": sorted(USERS)
})

# DIM_WEBSITE
sites = df[["website_name", "website_url"]].copy()
sites = sites.dropna(subset=["website_name"])
sites = sites.drop_duplicates(subset=["website_name"])
sites["website_id"] = range(1, len(sites) + 1)
sites["category"] = sites["website_name"].map({
    "YouTube": "Entertainment", "Netflix": "Entertainment", "Twitch": "Entertainment",
    "TikTok": "Social Media", "Instagram": "Social Media", "Facebook": "Social Media",
    "Twitter / X": "Social Media", "LinkedIn": "Social Media",
    "Google": "Search", "Wikipedia": "Reference",
    "Reddit": "Community", "Amazon": "Shopping",
    "GitHub": "Development", "Stack Overflow": "Development",
    "W3Schools": "Learning", "Coursera": "Learning", "Kaggle": "Learning",
    "ChatGPT": "AI Tools", "Spotify": "Music", "Canva": "Design"
}).fillna("Other")

# DIM_DATE
dates = df[["visit_date", "day_of_week", "month", "month_num", "week_of_year"]].copy()

dates = dates.drop_duplicates(subset=["visit_date"]).sort_values("visit_date")
dates["date_id"] = range(1, len(dates) + 1)
dates["quarter"] = ((dates["month_num"] - 1) // 3 + 1)

# FACT_VISITS — join keys back
df_merged = df.merge(users, on="user_name")
df_merged = df_merged.merge(sites[["website_name", "website_id"]], on="website_name")
df_merged = df_merged.merge(dates[["visit_date", "date_id"]], on="visit_date")

fact = df_merged[["user_id", "website_id", "date_id", "hour_of_day", "time_of_day", "visit_datetime"]].copy()
fact.insert(0, "visit_id", range(1, len(fact) + 1))


print(f"• dim_user    : {len(users)} rows")
print(f"• dim_website : {len(sites)} rows")
print(f"• dim_date    : {len(dates)} rows")
print(f"• fact_visits : {len(fact)} rows")


• dim_user    : 5 rows
• dim_website : 20 rows
• dim_date    : 61 rows
• fact_visits : 598 rows


In [11]:
import os
SAVE_DIR = "/kaggle/working"   

df.to_csv(os.path.join(SAVE_DIR, "cleaned_browsing_history.csv"), index=False)
users.to_csv(os.path.join(SAVE_DIR, "dim_user.csv"), index=False)
sites[["website_id", "website_name", "website_url", "category"]].to_csv(
    os.path.join(SAVE_DIR, "dim_website.csv"), index=False)
dates[["date_id", "visit_date", "day_of_week", "month", "month_num", "quarter", "week_of_year"]].to_csv(
    os.path.join(SAVE_DIR, "dim_date.csv"), index=False)
fact.to_csv(os.path.join(SAVE_DIR, "fact_visits.csv"), index=False)


print("   cleaned_browsing_history.csv")
print("   dim_user.csv")
print("   dim_website.csv")
print("   dim_date.csv")
print("   fact_visits.csv")


   cleaned_browsing_history.csv
   dim_user.csv
   dim_website.csv
   dim_date.csv
   fact_visits.csv


In [12]:
print("\n" + "=" * 60)
print("  ANALYTICS PREVIEW")
print("=" * 60)

clean = pd.read_csv(os.path.join(SAVE_DIR, "cleaned_browsing_history.csv"))

print("\n  Top 10 Most Visited Websites:")
top10 = clean["website_name"].value_counts().head(10)
for site, cnt in top10.items():
    print(f"    {site:<25} {cnt} visits")

print("\n  Visits per User:")
per_user = clean["user_name"].value_counts()
for user, cnt in per_user.items():
    print(f"    {user:<12} {cnt} visits")

print("\n  Most Active Time of Day:")
tod = clean["time_of_day"].value_counts()
for t, cnt in tod.items():
    print(f"    {t:<12} {cnt} visits")

print("\n  Visits per Month:")
months = clean.groupby(["month_num", "month"]).size().reset_index(name="count").sort_values("month_num")
for _, row in months.iterrows():
    print(f"    {row['month']:<12} {row['count']} visits")



  ANALYTICS PREVIEW

  Top 10 Most Visited Websites:
    Canva                     39 visits
    Instagram                 37 visits
    Facebook                  35 visits
    Twitter / X               34 visits
    GitHub                    34 visits
    Wikipedia                 33 visits
    Coursera                  31 visits
    YouTube                   31 visits
    Stack Overflow            29 visits
    Netflix                   29 visits

  Visits per User:
    Sara         136 visits
    Noor         122 visits
    Shahd        118 visits
    Bayan        113 visits
    Leen         109 visits

  Most Active Time of Day:
    Night        526 visits
    Morning      27 visits
    Evening      25 visits
    Afternoon    20 visits

  Visits per Month:
    January      20 visits
    September    17 visits
    October      22 visits
    November     18 visits
    December     18 visits
